# Homework 2: Vector Search


## Setup
Usage of lightweight ONNX `Embedder` (no PyTorch, no CUDA).


In [ ]:

import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from download import download
download("Xenova/all-MiniLM-L6-v2")

  exists models\Xenova\all-MiniLM-L6-v2\tokenizer.json
  exists models\Xenova\all-MiniLM-L6-v2\model.onnx


In [14]:
from embedder import Embedder

embedder = Embedder()
print("Embedder loaded successfully!")

Embedder loaded successfully!


## Q1. Embedding a Query

Embed the following query and report `v[0]`:




In [30]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector shape: {v.shape}")
print(f"v[0] = {v[0]:.4f}")
print(f"\nAnswer Q1: {round(v[0], 2)}")

Vector shape: (384,)
v[0] = -0.0206

Answer Q1: -0.02


## Loading the Data


In [16]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Number of documents: {len(documents)}")
print("Sample document keys:", list(documents[0].keys()) if documents else "none")

Number of documents: 72
Sample document keys: ['content', 'filename']


## Q2. Cosine Similarity

The embedder returns normalized vectors, so the dot product == cosine similarity.

Embed `02-vector-search/lessons/07-sqlitesearch-vector.md` and compute its cosine similarity with the Q1 query vector.


In [17]:
# Find the target document
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = None

for doc in documents:
    if target_filename in doc.get("filename", ""):
        target_doc = doc
        break

if target_doc:
    print(f"Found document: {target_doc['filename']}")
    print(f"Content length: {len(target_doc['content'])} chars")
else:
    print("Document not found. Available filenames:")
    for doc in documents[:5]:
        print(" ", doc.get('filename', 'N/A'))

Found document: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content length: 7219 chars


In [18]:
# Embed the target document's content
doc_vector = embedder.encode(target_doc["content"])

# Cosine similarity = dot product (since both vectors are normalized)
cosine_sim = v.dot(doc_vector)

print(f"Cosine similarity: {cosine_sim:.4f}")
print(f"\nAnswer Q2: {round(cosine_sim, 2)}")

Cosine similarity: 0.3611

Answer Q2: 0.36


## Q3. Chunking and Search by Hand

Chunk all pages, embed chunks, build a score matrix, and find the highest-scoring chunk for the Q1 query.


In [19]:
import numpy as np
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")
print("Sample chunk keys:", list(chunks[0].keys()) if chunks else "none")

Number of chunks: 295
Sample chunk keys: ['start', 'content', 'filename']


In [20]:
from tqdm import tqdm

# Embed all chunks using encode_batch (batch for efficiency)
BATCH_SIZE = 32
all_embeddings = []

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Embedding chunks"):
    batch = chunks[i:i+BATCH_SIZE]
    texts = [chunk["content"] for chunk in batch]
    batch_embeddings = embedder.encode_batch(texts)
    all_embeddings.append(batch_embeddings)

X = np.vstack(all_embeddings)
print(f"Embedding matrix shape: {X.shape}")

Embedding chunks: 100%|██████████| 10/10 [00:11<00:00,  1.16s/it]

Embedding matrix shape: (295, 384)


In [ ]:

scores = X.dot(v)

best_idx = scores.argmax()
best_chunk = chunks[best_idx]
best_score = scores[best_idx]

print(f"Highest score: {best_score:.4f}")
print(f"Best chunk filename: {best_chunk['filename']}")
print(f"\nAnswer Q3: {best_chunk['filename']}")

Highest score: 0.6489
Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md

Answer Q3: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector Search with minsearch




In [ ]:
from minsearch import VectorSearch
import numpy as np

vector_index = VectorSearch(
    keyword_fields=["filename"],
)

vector_index.fit(X, chunks)
print("VectorSearch index built!")


VectorSearch index built!


In [23]:
q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

vector_results = vector_index.search(
    query_vector=q4_vector,
    num_results=5
)

print("Top 5 results for Q4:")
for i, res in enumerate(vector_results):
    print(f"  {i+1}. {res['filename']}")

print(f"\nAnswer Q4: {vector_results[0]['filename']}")


Top 5 results for Q4:
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md

Answer Q4: 04-evaluation/lessons/05-search-metrics.md


## Q5. Text Search vs Vector Search




In [24]:
from minsearch import Index

# Build text search index
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)
print("Text Index built!")

Text Index built!


In [25]:
q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

# Vector search
q5_vector_results = vector_index.search(
    query_vector=q5_vector,
    num_results=5
)

# Text search
q5_text_results = text_index.search(
    query=q5_query,
    num_results=5
)

vector_filenames = set(r["filename"] for r in q5_vector_results)
text_filenames = set(r["filename"] for r in q5_text_results)

print("Vector search top 5 filenames:")
for r in q5_vector_results:
    print(f"  - {r['filename']}")

print("\nText search top 5 filenames:")
for r in q5_text_results:
    print(f"  - {r['filename']}")

only_in_vector = vector_filenames - text_filenames
print(f"\nFiles in vector results but NOT in text results: {only_in_vector}")
print(f"\nAnswer Q5: {only_in_vector}")


Vector search top 5 filenames:
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md

Text search top 5 filenames:
  - 02-vector-search/lessons/02-embeddings.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md

Files in vector results but NOT in text results: {'02-vector-search/lessons/08-pgvector.md'}

Answer Q5: {'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid Search with Reciprocal Rank Fusion (RRF)

Combine vector and text search results using RRF 

In [26]:
def rrf(result_lists, k=60, num_results=5):
    """Reciprocal Rank Fusion to combine multiple ranked lists."""
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [27]:
q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

# Vector search results (more results for better RRF)
q6_vector_results = vector_index.search(
    query_vector=q6_vector,
    num_results=10
)

# Text search results
q6_text_results = text_index.search(
    query=q6_query,
    num_results=10
)

# Apply RRF
results = rrf([q6_vector_results, q6_text_results])

print("Vector search top results:")
for i, r in enumerate(q6_vector_results[:5]):
    print(f"  {i+1}. {r['filename']}")

print("\nText search top results:")
for i, r in enumerate(q6_text_results[:5]):
    print(f"  {i+1}. {r['filename']}")

print("\nAfter RRF fusion:")
for i, r in enumerate(results):
    print(f"  {i+1}. {r['filename']}")

print(f"\nAnswer Q6: {results[0]['filename']}")


Vector search top results:
  1. 01-agentic-rag/lessons/01-intro.md
  2. 04-evaluation/lessons/02-ground-truth.md
  3. 01-agentic-rag/lessons/16-other-frameworks.md
  4. 01-agentic-rag/lessons/15-frameworks.md
  5. 01-agentic-rag/lessons/13-function-calling.md

Text search top results:
  1. 01-agentic-rag/lessons/14-agentic-loop.md
  2. 01-agentic-rag/lessons/13-function-calling.md
  3. 01-agentic-rag/lessons/13-function-calling.md
  4. 01-agentic-rag/lessons/13-function-calling.md
  5. 04-evaluation/lessons/02-ground-truth.md

After RRF fusion:
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/13-function-calling.md
  3. 01-agentic-rag/lessons/01-intro.md
  4. 01-agentic-rag/lessons/14-agentic-loop.md
  5. 04-evaluation/lessons/02-ground-truth.md

Answer Q6: 01-agentic-rag/lessons/13-function-calling.md


## Summary of Answers


In [29]:

print("HOMEWORK 2 ANSWERS SUMMARY")
print(f"Q1 - v[0] of query embedding: {round(v[0], 2)}")
print(f"Q2 - Cosine similarity: {round(cosine_sim, 2)}")
print(f"Q3 - Highest scoring chunk filename: {best_chunk['filename']}")
print(f"Q4 - First result filename: {vector_results[0]['filename']}")
print(f"Q5 - In vector but not text: {only_in_vector}")
print(f"Q6 - First result after RRF: {results[0]['filename']}")

HOMEWORK 2 ANSWERS SUMMARY
Q1 - v[0] of query embedding: -0.02
Q2 - Cosine similarity: 0.36
Q3 - Highest scoring chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Q4 - First result filename: 04-evaluation/lessons/05-search-metrics.md
Q5 - In vector but not text: {'02-vector-search/lessons/08-pgvector.md'}
Q6 - First result after RRF: 01-agentic-rag/lessons/13-function-calling.md
